# LeetCode 2114 — Maximum Number of Words Found in Sentences bằng Python

**Link đề:** https://leetcode.com/problems/maximum-number-of-words-found-in-sentences/
**Độ khó:** Easy · **Chủ đề:** Array, String

```python
class Solution:
    def mostWordsFound(self, sentences: list[str]) -> int:
        ...
```

## Mục tiêu học
1. Đếm số từ của một câu theo nhiều cách: `split()`, **đếm dấu cách + 1**, tự quét ký tự (máy trạng thái "đang trong từ").
2. Mẫu **tìm giá trị lớn nhất** khi duyệt: biến `best`, hàm `max`, `max(..., key=...)`, `map`.
3. Hiểu vì sao "số dấu cách + 1" đúng **nhờ ràng buộc đề** (một dấu cách, không có dấu cách đầu/cuối) — và nó sai khi bỏ ràng buộc.
4. Trace, chứng minh, Time/Space, test đối chiếu, lỗi hay gặp, biến thể, bài tập.

## 1. Đề bài và cách hiểu đúng

Một **câu** là danh sách các từ cách nhau **đúng một dấu cách**, **không có** dấu cách ở đầu hoặc cuối.

Cho mảng `sentences`. Trả về **số từ lớn nhất** trong một câu.

| `sentences` | Số từ từng câu | Kết quả |
|---|---|:---:|
| `["alice and bob love leetcode", "i think so too", "this is great thanks very much"]` | 5, 4, 6 | 6 |
| `["please wait", "continue to fight", "continue to win"]` | 2, 3, 3 | 3 |

**Constraints:** `1 <= len(sentences) <= 100`; mỗi câu dài 1..100, chỉ chữ thường và dấu cách; không có dấu cách đầu/cuối; các từ cách nhau **một** dấu cách.

Bài tách thành hai việc nhỏ: (1) **đếm số từ** của một câu; (2) **lấy max** qua các câu.

In [1]:
examples = [
    (["alice and bob love leetcode", "i think so too", "this is great thanks very much"], 6),
    (["please wait", "continue to fight", "continue to win"], 3),
]
for ss, exp in examples:
    print(ss, "->", exp)

['alice and bob love leetcode', 'i think so too', 'this is great thanks very much'] -> 6
['please wait', 'continue to fight', 'continue to win'] -> 3


## 2. Nền tảng Python cần dùng

### 2.1. `str.split()` — tách câu thành list các từ

- `s.split()` (không đối số) tách theo **mọi khoảng trắng** và **bỏ** các phần rỗng.
- `s.split(" ")` tách theo **đúng một dấu cách**; nếu có hai dấu cách liền nhau sẽ sinh chuỗi rỗng `""`.

In [2]:
s = "i think so too"
print(s.split())            # ['i', 'think', 'so', 'too']
print(len(s.split()))       # 4
t = "a  b"                  # hai dấu cách
print(t.split(), t.split(" "))

['i', 'think', 'so', 'too']
4
['a', 'b'] ['a', '', 'b']


### 2.2. `str.count(sub)` — đếm số lần chuỗi con xuất hiện

In [3]:
s = "alice and bob love leetcode"
print(s.count(" "))         # 4 dấu cách
print(s.count(" ") + 1)     # 5 từ
print("banana".count("an")) # 2

4
5
2


### 2.3. Mẫu "giữ giá trị lớn nhất"

Khởi tạo `best = 0` (số từ luôn ≥ 1 nên 0 an toàn), duyệt và cập nhật khi gặp giá trị lớn hơn.

In [4]:
values = [5, 4, 6, 2]
best = 0
for v in values:
    if v > best:
        best = v
    print(f"v={v}, best={best}")

v=5, best=5
v=4, best=5
v=6, best=6
v=2, best=6


### 2.4. `max(iterable)`, `max(..., key=...)`, `map(func, iterable)`

- `max([5, 4, 6])` → 6.
- `max(sentences, key=f)` trả **phần tử** có `f` lớn nhất (không phải giá trị `f`).
- `map(f, data)` áp `f` lên từng phần tử (lười — tạo dần).

In [5]:
words_count = lambda s: s.count(" ") + 1
ss = ["please wait", "continue to fight", "continue to win"]
print(list(map(words_count, ss)))      # [2, 3, 3]
print(max(map(words_count, ss)))       # 3
print(max(ss, key=words_count))        # câu đầu tiên có nhiều từ nhất

[2, 3, 3]
3
continue to fight


### 2.5. Duyệt từng ký tự và so sánh với `" "`

In [6]:
for i, ch in enumerate("hi yo"):
    print(i, repr(ch), ch == " ")

0 'h' False
1 'i' False
2 ' ' True
3 'y' False
4 'o' False


## 3. Cách 1 — Trực giác: tự tách từng từ bằng tay

Người mới chưa biết `split` có thể tự xây từng từ: duyệt ký tự, nếu là chữ thì nối vào `current`; gặp dấu cách thì "đóng" từ hiện tại. Cuối câu nhớ đóng từ cuối.

In [7]:
def split_manual(s: str) -> list[str]:
    words, current = [], ""
    for ch in s:
        if ch == " ":
            if current:
                words.append(current)
            current = ""
        else:
            current += ch
    if current:                     # từ cuối cùng không có dấu cách phía sau
        words.append(current)
    return words

def most_words_manual(sentences: list[str]) -> int:
    best = 0
    for s in sentences:
        best = max(best, len(split_manual(s)))
    return best

print(split_manual("alice and bob"))
for ss, exp in examples:
    print(most_words_manual(ss), "kỳ vọng", exp)

['alice', 'and', 'bob']
6 kỳ vọng 6
3 kỳ vọng 3


## 4. Nút thắt — ta có cần **danh sách các từ** không?

Đề chỉ hỏi **số lượng** từ. Việc tạo list từ (dù bằng tay hay `split()`) tốn thêm bộ nhớ `O(L)` và thời gian nối chuỗi.

Gợi ý của đề: *"count the number of words by looking for the number of space characters and adding it by 1."*

Vì các từ cách nhau **đúng một** dấu cách và không có dấu cách ở hai đầu:

```text
"alice and bob love leetcode"
      ^   ^   ^    ^            4 dấu cách  ->  5 từ
```

`k` từ thì có đúng `k - 1` khoảng giữa ⇒ **số từ = số dấu cách + 1**.

In [8]:
for s in ["hello", "a b", "alice and bob love leetcode"]:
    print(f"{s!r:32} dấu cách={s.count(' ')}, số từ={s.count(' ') + 1}, split={len(s.split())}")

'hello'                          dấu cách=0, số từ=1, split=1
'a b'                            dấu cách=1, số từ=2, split=2
'alice and bob love leetcode'    dấu cách=4, số từ=5, split=5


## 5. Bước đột phá — Đếm "ranh giới" thay vì đếm "đối tượng"

Đây là ý tưởng tổng quát: số **phần tử** trong một dãy = số **vách ngăn** + 1 (giống "n cây cột cần n−1 khoảng"). Đếm vách ngăn không cần dựng lại từng phần tử.

Cách tổng quát hơn (không cần ràng buộc dấu cách): đếm số vị trí **bắt đầu một từ** — ký tự khác dấu cách mà đứng **đầu câu** hoặc **sau một dấu cách**.

In [9]:
def count_word_starts(s):
    return sum(1 for i, ch in enumerate(s) if ch != " " and (i == 0 or s[i - 1] == " "))

for s in ["alice and bob", "  lots   of   spaces  ", "x"]:
    print(f"{s!r:26} -> {count_word_starts(s)} từ (split cho {len(s.split())})")

'alice and bob'            -> 3 từ (split cho 3)
'  lots   of   spaces  '   -> 3 từ (split cho 3)
'x'                        -> 1 từ (split cho 1)


## 6. Lời giải chính — mổ xẻ rồi ghép

### 6.1. Đoạn 1: đếm từ của một câu

In [10]:
def count_words(sentence: str) -> int:
    return sentence.count(" ") + 1

print(count_words("i think so too"))

4


### 6.2. Đoạn 2: lấy max qua mọi câu

In [11]:
def most_words_found(sentences: list[str]) -> int:
    best = 0
    for s in sentences:
        best = max(best, count_words(s))
    return best

for ss, exp in examples:
    print(most_words_found(ss), "kỳ vọng", exp)

6 kỳ vọng 6
3 kỳ vọng 3


### 6.3. Gọn trong một dòng

`max(s.count(" ") for s in sentences) + 1` — cộng 1 **sau** khi lấy max cũng được, vì `max(a+1, b+1) = max(a, b) + 1`.

In [12]:
for ss, exp in examples:
    print(max(s.count(" ") for s in ss) + 1, "kỳ vọng", exp)

6 kỳ vọng 6
3 kỳ vọng 3


## 7. TRACE từng bước

### 7.1. Bảng trace — ví dụ 1

| Câu | Số dấu cách | Số từ | best trước | best sau |
|---|:---:|:---:|:---:|:---:|
| `"alice and bob love leetcode"` | 4 | 5 | 0 | 5 |
| `"i think so too"` | 3 | 4 | 5 | 5 |
| `"this is great thanks very much"` | 5 | 6 | 5 | 6 |

In [13]:
def trace_most_words(sentences):
    best = 0
    for idx, s in enumerate(sentences):
        spaces = s.count(" ")
        words = spaces + 1
        before = best
        best = max(best, words)
        print(f"[{idx}] {s!r:34} spaces={spaces} words={words} best: {before} -> {best}")
    return best

print("Kết quả:", trace_most_words(examples[0][0]))

[0] 'alice and bob love leetcode'      spaces=4 words=5 best: 0 -> 5
[1] 'i think so too'                   spaces=3 words=4 best: 5 -> 5
[2] 'this is great thanks very much'   spaces=5 words=6 best: 5 -> 6
Kết quả: 6


### 7.2. Trace cách tự quét ký tự (đếm dấu cách bằng tay)

In [14]:
def trace_scan(sentence):
    spaces = 0
    for i, ch in enumerate(sentence):
        if ch == " ":
            spaces += 1
            print(f"  i={i}: gặp dấu cách -> spaces={spaces}")
    print(f"  => {spaces} dấu cách, {spaces + 1} từ")
    return spaces + 1

trace_scan("i think so too")

  i=1: gặp dấu cách -> spaces=1
  i=7: gặp dấu cách -> spaces=2
  i=10: gặp dấu cách -> spaces=3
  => 3 dấu cách, 4 từ


4

### 7.3. Trường hợp biên: câu một từ, và nhiều câu bằng nhau

In [15]:
print("Kết quả:", trace_most_words(["hello", "world"]))
print("Kết quả:", trace_most_words(["please wait", "continue to fight", "continue to win"]))

[0] 'hello'                            spaces=0 words=1 best: 0 -> 1
[1] 'world'                            spaces=0 words=1 best: 1 -> 1
Kết quả: 1
[0] 'please wait'                      spaces=1 words=2 best: 0 -> 2
[1] 'continue to fight'                spaces=2 words=3 best: 2 -> 3
[2] 'continue to win'                  spaces=2 words=3 best: 3 -> 3
Kết quả: 3


## 8. Code nộp LeetCode (Python 3) — nhiều cách

### Cách A — `split()` + `len`

In [16]:
class Solution:
    def mostWordsFound(self, sentences: list[str]) -> int:
        return max(len(s.split()) for s in sentences)

print([Solution().mostWordsFound(ss) for ss, _ in examples])

[6, 3]


### Cách B — Đếm dấu cách + 1 (lời giải chính, theo gợi ý)

In [17]:
class Solution:
    def mostWordsFound(self, sentences: list[str]) -> int:
        return max(s.count(" ") for s in sentences) + 1

print([Solution().mostWordsFound(ss) for ss, _ in examples])

[6, 3]


### Cách C — Vòng lặp tường minh (không dùng hàm có sẵn)

In [18]:
class Solution:
    def mostWordsFound(self, sentences: list[str]) -> int:
        best = 0
        for s in sentences:
            words = 1
            for ch in s:
                if ch == " ":
                    words += 1
            if words > best:
                best = words
        return best

print([Solution().mostWordsFound(ss) for ss, _ in examples])

[6, 3]


### Cách D — Đếm vị trí bắt đầu từ (đúng cả khi có nhiều dấu cách)

In [19]:
class Solution:
    def mostWordsFound(self, sentences: list[str]) -> int:
        def words(s: str) -> int:
            return sum(1 for i, ch in enumerate(s)
                       if ch != " " and (i == 0 or s[i - 1] == " "))
        return max(map(words, sentences))

print([Solution().mostWordsFound(ss) for ss, _ in examples])

[6, 3]


### Cách E — Máy trạng thái "đang ở trong từ" (in_word)

Biến `in_word` cho biết ký tự trước có thuộc một từ không. Mỗi lần chuyển từ "ngoài" sang "trong" là bắt đầu một từ mới → cộng 1. Kỹ thuật này dùng nhiều trong xử lý văn bản.

In [20]:
class Solution:
    def mostWordsFound(self, sentences: list[str]) -> int:
        best = 0
        for s in sentences:
            count, in_word = 0, False
            for ch in s:
                if ch != " " and not in_word:
                    count += 1
                    in_word = True
                elif ch == " ":
                    in_word = False
            best = max(best, count)
        return best

print([Solution().mostWordsFound(ss) for ss, _ in examples])

[6, 3]


## 9. Vì sao thuật toán đúng?

**Số từ = số dấu cách + 1 (dưới ràng buộc đề):** câu gồm `k ≥ 1` từ `w1 w2 ... wk` nối bằng đúng một dấu cách giữa hai từ liên tiếp, không có dấu cách ở đầu/cuối, và từ không chứa dấu cách. Vậy câu có đúng `k − 1` dấu cách ⇒ `k = spaces + 1`.

**Lấy max:** invariant — sau khi xử lý `j` câu đầu, `best` = số từ lớn nhất trong `j` câu đó (vì `best = max(best, words_j)`). Khởi tạo `best = 0` hợp lệ vì mọi câu có ≥ 1 từ. Kết thúc: `best` là max trên toàn bộ.

**Cách D/E** đúng cả khi không có ràng buộc: mỗi từ có **đúng một** ký tự bắt đầu (ký tự khác cách mà trước nó là dấu cách hoặc đầu câu) ⇒ đếm ký tự bắt đầu = đếm từ.

## 10. Phân tích Time / Space Complexity

Gọi `m = len(sentences)`, `L` = độ dài tối đa của câu, `S` = tổng độ dài các câu.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| Tự tách bằng tay | `O(S)` (+ chi phí nối chuỗi) | `O(L)` | baseline |
| A. `split()` | `O(S)` | `O(L)` list từ | dễ đọc, đúng cả khi nhiều dấu cách |
| B. `count(" ") + 1` | `O(S)` | `O(1)` | lời giải chính, dựa vào ràng buộc |
| C. Vòng lặp tay | `O(S)` | `O(1)` | không dùng thư viện |
| D. Đếm ký tự bắt đầu | `O(S)` | `O(1)` | tổng quát |
| E. Máy trạng thái | `O(S)` | `O(1)` | tổng quát, dễ mở rộng |

Mọi cách đều phải đọc toàn bộ input nên `Ω(S)` là giới hạn dưới.

## 11. Kiểm thử: mẫu đề, biên, phản ví dụ, đối chiếu

In [21]:
import random

def sol_manual(ss): return most_words_manual(ss)
def sol_A(ss): return max(len(s.split()) for s in ss)
def sol_B(ss): return max(s.count(" ") for s in ss) + 1
def sol_C(ss): return most_words_found(ss)
def sol_D(ss): return max(count_word_starts(s) for s in ss)
def sol_E(ss):
    best = 0
    for s in ss:
        c, inw = 0, False
        for ch in s:
            if ch != " " and not inw: c += 1; inw = True
            elif ch == " ": inw = False
        best = max(best, c)
    return best

SOLS = [sol_manual, sol_A, sol_B, sol_C, sol_D, sol_E]

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:28} PASS ({actual})")

tests = examples + [
    (["a"], 1),                               # một câu một từ
    (["a b c d e f g h i j"], 10),
    (["x", "x y", "x"], 2),                   # max ở giữa
    (["a " * 49 + "a"], 50),                  # 50 từ, dài 99
]
for i, (ss, exp) in enumerate(tests):
    for f in SOLS:
        check(f"test {i} {f.__name__}", f(ss), exp)

test 0 sol_manual            PASS (6)
test 0 sol_A                 PASS (6)
test 0 sol_B                 PASS (6)
test 0 sol_C                 PASS (6)
test 0 sol_D                 PASS (6)
test 0 sol_E                 PASS (6)
test 1 sol_manual            PASS (3)
test 1 sol_A                 PASS (3)
test 1 sol_B                 PASS (3)
test 1 sol_C                 PASS (3)
test 1 sol_D                 PASS (3)
test 1 sol_E                 PASS (3)
test 2 sol_manual            PASS (1)
test 2 sol_A                 PASS (1)
test 2 sol_B                 PASS (1)
test 2 sol_C                 PASS (1)
test 2 sol_D                 PASS (1)
test 2 sol_E                 PASS (1)
test 3 sol_manual            PASS (10)
test 3 sol_A                 PASS (10)
test 3 sol_B                 PASS (10)
test 3 sol_C                 PASS (10)
test 3 sol_D                 PASS (10)
test 3 sol_E                 PASS (10)
test 4 sol_manual            PASS (2)
test 4 sol_A                 PASS (2)
test 4

In [22]:
rng = random.Random(2114)
def rand_sentence():
    return " ".join("".join(rng.choice("abc") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 8)))
for _ in range(500):
    ss = [rand_sentence() for _ in range(rng.randint(1, 5))]
    outs = [f(ss) for f in SOLS]
    assert len(set(outs)) == 1, (ss, outs)
print("500 test ngẫu nhiên (đúng ràng buộc đề): 6 cách khớp nhau")

500 test ngẫu nhiên (đúng ràng buộc đề): 6 cách khớp nhau


In [23]:
# Ngoài ràng buộc: nhiều dấu cách -> cách B (count+1) khác các cách tổng quát
messy = ["  hello   world  "]
print("count+1 :", sol_B(messy), "| split:", sol_A(messy), "| đếm ký tự bắt đầu:", sol_D(messy))

count+1 : 8 | split: 2 | đếm ký tự bắt đầu: 2


## 12. Lỗi người mới hay gặp — code sai, vì sao sai, cách sửa

### Lỗi 1: Quên `+ 1` (trả số dấu cách)

In [24]:
def wrong_no_plus1(sentences):
    return max(s.count(" ") for s in sentences)   # SAI

print("Sai :", wrong_no_plus1(examples[0][0]), "| Đúng:", sol_B(examples[0][0]))

Sai : 5 | Đúng: 6


### Lỗi 2: Đếm tổng số từ của **mọi** câu thay vì max

In [25]:
def wrong_sum(sentences):
    return sum(len(s.split()) for s in sentences)   # SAI: đề hỏi max trong MỘT câu

print("Sai :", wrong_sum(examples[1][0]), "| Đúng:", sol_A(examples[1][0]))

Sai : 8 | Đúng: 3


### Lỗi 3: `max(sentences, key=...)` rồi trả luôn — ra **câu**, không ra **số**

In [26]:
res = max(examples[0][0], key=lambda s: s.count(" "))
print("Sai :", repr(res), "(đây là câu, không phải số)")
print("Đúng:", res.count(" ") + 1)

Sai : 'this is great thanks very much' (đây là câu, không phải số)
Đúng: 6


### Lỗi 4: Dùng `len(s)` (số ký tự) thay vì số từ

In [27]:
def wrong_len(sentences):
    return max(len(s) for s in sentences)   # SAI: đếm ký tự

print("Sai :", wrong_len(examples[0][0]), "| Đúng:", sol_B(examples[0][0]))

Sai : 30 | Đúng: 6


### Lỗi 5: Dùng `split(" ")` khi input có nhiều dấu cách (ngoài đề này) → đếm cả chuỗi rỗng

In [28]:
s = "a  b"
print("split(' '):", s.split(" "), "->", len(s.split(" ")), "| split():", s.split(), "->", len(s.split()))

split(' '): ['a', '', 'b'] -> 3 | split(): ['a', 'b'] -> 2


## 13. Mẹo nhận dạng pattern

- "Số phần tử được ngăn bởi dấu phân cách đơn" → **số dấu phân cách + 1** (không cần tách).
- "Max/min của một đại lượng tính trên từng phần tử" → `max(f(x) for x in data)`.
- Văn bản lộn xộn (nhiều dấu cách) → `split()` không đối số hoặc **máy trạng thái in_word**.
- Luôn đọc **constraints**: mẹo `count + 1` chỉ đúng nhờ ràng buộc "đúng một dấu cách, không đầu/cuối".

Bài liên quan: 58 Length of Last Word, 434 Number of Segments in a String, 1816 Truncate Sentence, 2129 Capitalize the Title.

## 14. Biến thể

### 14.1. Trả về **câu** có nhiều từ nhất (bằng nhau lấy câu đầu)

In [29]:
def sentence_with_most_words(sentences):
    return max(sentences, key=lambda s: s.count(" "))

print(repr(sentence_with_most_words(examples[1][0])))

'continue to fight'


### 14.2. Đếm số "đoạn" (segment) trong chuỗi có thể có nhiều dấu cách (LeetCode 434)

In [30]:
def count_segments(s):
    return len(s.split())

print(count_segments(", , , ,        a, eaefa"), count_segments(""))

6 0


### 14.3. Câu có **nhiều từ khác nhau** nhất

In [31]:
def most_distinct_words(sentences):
    return max(len(set(s.split())) for s in sentences)

print(most_distinct_words(["a a a a", "a b c"]))

3


## 15. Hệ thống luyện tập — 6 bài từ dễ đến nâng cao

Đáp án ở **mục 17**.

### Bài 1 — Đếm số dấu cách bằng vòng lặp (không dùng `count`)

In [32]:
def count_spaces(s):
    # TODO: viết code ở đây
    pass

# assert count_spaces("i think so too") == 3

### Bài 2 — Tự viết lại lời giải #2114

In [33]:
def most_words_ex(sentences):
    # TODO: viết code ở đây
    pass

# assert most_words_ex(["please wait", "continue to fight", "continue to win"]) == 3

### Bài 3 — Độ dài từ cuối cùng (LeetCode 58), có thể có dấu cách ở cuối

In [34]:
def length_of_last_word(s):
    # TODO: viết code ở đây
    pass

# assert length_of_last_word("Hello World") == 5
# assert length_of_last_word("   fly me   to   the moon  ") == 4

### Bài 4 — Biến thể: số từ **ít nhất** trong một câu

In [35]:
def fewest_words(sentences):
    # TODO: viết code ở đây
    pass

# assert fewest_words(["alice and bob", "hi", "a b c d"]) == 1

### Bài 5 — Tự tạo test làm lộ **Lỗi 4** (`len(s)` thay vì số từ)
Tìm các mảng mà `wrong_len` **vô tình đúng** và các mảng làm nó sai.

In [36]:
accidental_ok = [
    # TODO: mảng mà wrong_len(ss) == sol_B(ss)
]
exposing = [
    # TODO: mảng mà wrong_len(ss) != sol_B(ss)
]
# for ss in accidental_ok: assert wrong_len(ss) == sol_B(ss)
# for ss in exposing: assert wrong_len(ss) != sol_B(ss)

### Bài 6 — Mini challenge: truncate sentence (LeetCode 1816)
Giữ lại `k` từ đầu tiên của câu `s` (đúng ràng buộc một dấu cách). Thử làm **không dùng `split`**: tìm vị trí dấu cách thứ `k`.

In [37]:
def truncate_sentence(s, k):
    # TODO: viết code ở đây
    pass

# assert truncate_sentence("Hello how are you Contestant", 4) == "Hello how are you"
# assert truncate_sentence("chopper is not a tanuki", 5) == "chopper is not a tanuki"

## 16. Gợi ý ba mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | duyệt từng ký tự | đếm khi `ch == " "` | `sum(ch == " " for ch in s)` |
| 2 | đếm từ một câu | dấu cách + 1 | `max(...) + 1` |
| 3 | bỏ dấu cách cuối trước | `s.rstrip()` hoặc duyệt ngược | `len(s.split()[-1])` |
| 4 | như lời giải nhưng `min` | `min(s.count(" ") for s in ...)` | `+ 1` |
| 5 | `len` = số ký tự ≥ số từ | bằng nhau khi câu chỉ có 1 ký tự | `["a"]` trùng; `["ab"]` làm lộ |
| 6 | dấu cách thứ k là ranh giới | duyệt và đếm dấu cách | cắt `s[:i]` khi đếm đủ k |

## 17. Đáp án tham khảo

In [38]:
def count_spaces(s):
    c = 0
    for ch in s:
        if ch == " ":
            c += 1
    return c

def most_words_ex(sentences):
    return max(count_spaces(s) for s in sentences) + 1

def length_of_last_word(s):
    i = len(s) - 1
    while i >= 0 and s[i] == " ":
        i -= 1
    length = 0
    while i >= 0 and s[i] != " ":
        length += 1; i -= 1
    return length

def fewest_words(sentences):
    return min(s.count(" ") for s in sentences) + 1

accidental_ok = [["a"], ["b", "c"]]
exposing = [examples[0][0], ["ab"], ["a b"]]

def truncate_sentence(s, k):
    spaces = 0
    for i, ch in enumerate(s):
        if ch == " ":
            spaces += 1
            if spaces == k:
                return s[:i]
    return s

assert count_spaces("i think so too") == 3
assert most_words_ex(["please wait", "continue to fight", "continue to win"]) == 3
assert length_of_last_word("Hello World") == 5
assert length_of_last_word("   fly me   to   the moon  ") == 4
assert fewest_words(["alice and bob", "hi", "a b c d"]) == 1
for ss in accidental_ok: assert wrong_len(ss) == sol_B(ss)
for ss in exposing: assert wrong_len(ss) != sol_B(ss)
assert truncate_sentence("Hello how are you Contestant", 4) == "Hello how are you"
assert truncate_sentence("chopper is not a tanuki", 5) == "chopper is not a tanuki"
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


**Giải thích bằng lời:**
- Bài 1–2: đếm vách ngăn rồi + 1.
- Bài 3: bỏ khoảng trắng cuối, rồi đếm ký tự cho tới dấu cách gần nhất — duyệt ngược, `O(độ dài từ cuối + số dấu cách cuối)`.
- Bài 4: đối xứng với lời giải chính.
- Bài 5: mỗi câu luôn có `len(s) >= số từ`, dấu bằng chỉ xảy ra khi câu là **một từ một ký tự** (ví dụ `"a"`). Vì vậy `wrong_len` đúng khi mọi câu đều là một ký tự; chỉ cần một câu dài ≥ 2 ký tự là max bị đẩy lên → lộ lỗi.
- Bài 6: dấu cách thứ `k` là ranh giới sau từ thứ `k`; không đủ `k` dấu cách thì câu có ≤ k từ → giữ nguyên.

## 18. Tổng kết — template + checklist

**Template:**
```python
return max(s.count(" ") for s in sentences) + 1     # nhờ ràng buộc đề
# tổng quát:  max(len(s.split()) for s in sentences)
```

**Checklist trước khi submit:**
- [ ] Đọc ràng buộc: một dấu cách, không đầu/cuối ⇒ `count + 1` đúng.
- [ ] Nhớ `+ 1`.
- [ ] Trả **max trong một câu**, không phải tổng.
- [ ] Trả **số**, không phải câu.
- [ ] Đúng tên method `mostWordsFound`.
- [ ] Chạy lại 2 ví dụ: `6`, `3`.